## 04 — Data Transformation and AOI Mapping

In [ ]:
### Prepare variables and map fixations to analytical regions

In [105]:
import pandas as pd
from pathlib import Path


In [107]:
#Get the main project folder
project_folder = Path.cwd().parent



In [109]:
# Construct the correct path to the interim folder
csv_file_path = project_folder / "data" / "interim" / "et_data_joined_03.csv"

In [111]:
# Read the dataset into pandas
et_data_joined_03 = pd.read_csv(csv_file_path)

In [112]:
et_data_joined = et_data_joined_03.copy()

In [115]:
print(et_data_joined.shape)

(371510, 30)


In [117]:
print(et_data_joined.head())

  ParticipantName    items  Lists  RecordingTimestamp  time_diff  \
0          part05  Item12c  List2              139899          0   
1          part05  Item12c  List2              139907          8   
2          part05  Item12c  List2              139915         16   
3          part05  Item12c  List2              139924         25   
4          part05  Item12c  List2              139932         33   

   GazeEventDur_ms Validity left Validity right EyeMovementType  GazePointX  \
0              225         Valid          Valid        Fixation       477.0   
1              225         Valid          Valid        Fixation       482.0   
2              225         Valid          Valid        Fixation       476.0   
3              225         Valid          Valid        Fixation       485.0   
4              225         Valid          Valid        Fixation       495.0   

   ...        audio Pictures  FirstMention  subj_image  obj_image  \
0  ...  item12c.wav      l_r          left     

In [119]:
et_data_joined.head()

,ParticipantName,items,Lists,RecordingTimestamp,time_diff,GazeEventDur_ms,Validity left,Validity right,EyeMovementType,GazePointX,...,audio,Pictures,FirstMention,subj_image,obj_image,middle_image,image_left,image_middle,image_right,sentences
0,part05,Item12c,List2,139899,0,225,Valid,Valid,Fixation,477.0,...,item12c.wav,l_r,left,hawk,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...
1,part05,Item12c,List2,139907,8,225,Valid,Valid,Fixation,482.0,...,item12c.wav,l_r,left,hawk,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...
2,part05,Item12c,List2,139915,16,225,Valid,Valid,Fixation,476.0,...,item12c.wav,l_r,left,hawk,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...
3,part05,Item12c,List2,139924,25,225,Valid,Valid,Fixation,485.0,...,item12c.wav,l_r,left,hawk,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...
4,part05,Item12c,List2,139932,33,225,Valid,Valid,Fixation,495.0,...,item12c.wav,l_r,left,hawk,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...


In [121]:
#Name: AOI, Length: 371510, dtype: object
print(et_data_joined["AOI"])
nan_count = et_data_joined["AOI"].isna().sum()
print(f"Number of NaN values in AOI column: {nan_count}")
# Number of NaN values in AOI column: 185539 - which will be filled with 'out' in the next step. 
# These are the fixations outside the marked areas of interest - AOIs

0         left
1         left
2         left
3         left
4         left
          ... 
371505    left
371506    left
371507    left
371508    left
371509    left
Name: AOI, Length: 371510, dtype: object
Number of NaN values in AOI column: 185539


In [123]:
# Fill in formal NaN / None values with 'out'; these are the fixations outsed the AOIs
et_data_joined["AOI"] = et_data_joined["AOI"].fillna("out")

In [125]:
# Remove observations with missing gaze coordinates to ensure that only
# records with valid X and Y positions are retained for spatial analysis.

et_data_joined = et_data_joined.dropna(subset=["GazePointX", "GazePointY"]).copy()

# Report the number of observations remaining after cleaning.
print(f"Remaining rows: {len(et_data_joined):,}")

# The dataset decreased from 371,510 to 371,169 rows, indicating that
# 341 observations with missing gaze coordinates were excluded.

Remaining rows: 371,169


In [127]:
import numpy as np
import pandas as pd

# Next to add AOI binary columns (left, right)
et_data_joined["left"] = np.where(et_data_joined["AOI"] == "left", 1, 0)
et_data_joined["right"] = np.where(et_data_joined["AOI"] == "right", 1, 0)
et_data_joined["out"] = np.where(et_data_joined["AOI"] == "out", 2, 2)

In [129]:
# Populate Condition column using the existing 'svo_ovs' metadata column to get 'SVO' / 'OVS'
et_data_joined["Condition"] = et_data_joined["svo_ovs"].astype(str).str.upper()

In [131]:
et_data_joined.head()

,ParticipantName,items,Lists,RecordingTimestamp,time_diff,GazeEventDur_ms,Validity left,Validity right,EyeMovementType,GazePointX,...,obj_image,middle_image,image_left,image_middle,image_right,sentences,left,right,out,Condition
0,part05,Item12c,List2,139899,0,225,Valid,Valid,Fixation,477.0,...,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS
1,part05,Item12c,List2,139907,8,225,Valid,Valid,Fixation,482.0,...,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS
2,part05,Item12c,List2,139915,16,225,Valid,Valid,Fixation,476.0,...,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS
3,part05,Item12c,List2,139924,25,225,Valid,Valid,Fixation,485.0,...,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS
4,part05,Item12c,List2,139932,33,225,Valid,Valid,Fixation,495.0,...,fox,trees,hawk,trees,fox,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS


In [133]:
aoi1_conditions = [
    et_data_joined["image_left"] == et_data_joined["subj_image"],
    et_data_joined["image_left"] == et_data_joined["obj_image"],
]

aoi1_choices = ["Subject", "Object"]
et_data_joined["AOI1"] = np.select(aoi1_conditions, aoi1_choices, default=None)


In [135]:

aoi2_conditions = [
    et_data_joined["image_right"] == et_data_joined["subj_image"],
    et_data_joined["image_right"] == et_data_joined["obj_image"],
]
aoi2_choices = ["Subject", "Object"]
et_data_joined["AOI2"] = np.select(aoi2_conditions, aoi2_choices, default=None)

In [137]:
subject_condition = (
    (et_data_joined["AOI1"] == "Subject") & (et_data_joined["left"] == 1)
) | (
    (et_data_joined["AOI2"] == "Subject") & (et_data_joined["right"] == 1)
)

object_condition = (
    (et_data_joined["AOI1"] == "Object") & (et_data_joined["left"] == 1)
) | (
    (et_data_joined["AOI2"] == "Object") & (et_data_joined["right"] == 1)
)

In [139]:
et_data_joined["Subject"] = np.where(subject_condition, 1, 0)
et_data_joined["Object"] = np.where(object_condition, 1, 0)

In [141]:
roi_conditions = [
    et_data_joined["Subject"] == 1,
    et_data_joined["Object"] == 1
]
roi_choices = ["Subject", "Object"]
et_data_joined["ROI"] = np.select(roi_conditions, roi_choices, default="out") # ROI = Region of Interest

In [143]:
et_data_joined.head()

,ParticipantName,items,Lists,RecordingTimestamp,time_diff,GazeEventDur_ms,Validity left,Validity right,EyeMovementType,GazePointX,...,sentences,left,right,out,Condition,AOI1,AOI2,Subject,Object,ROI
0,part05,Item12c,List2,139899,0,225,Valid,Valid,Fixation,477.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
1,part05,Item12c,List2,139907,8,225,Valid,Valid,Fixation,482.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
2,part05,Item12c,List2,139915,16,225,Valid,Valid,Fixation,476.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
3,part05,Item12c,List2,139924,25,225,Valid,Valid,Fixation,485.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject
4,part05,Item12c,List2,139932,33,225,Valid,Valid,Fixation,495.0,...,მელას დასცინის ქორი ხეებს შორის. ის არის გაბრ...,1,0,2,OVS,Subject,Object,1,0,Subject


In [145]:
# Create an independent copy of et_data_joined
et_data_transformed_04 = et_data_joined.copy()

In [147]:
# Save the transformed dataset to the interim data directory
# to preserve an intermediate analysis-ready version for subsequent steps.
interim_folder = project_folder / "data" / "interim"

csv_file_path = interim_folder / "et_data_transformed_04.csv"
et_data_transformed_04.to_csv(csv_file_path, index=False)



In [148]:
# End of the stucturing, editing and checking the eye-tracking file with the metadata. The next step is to aggrigation the data for analysis.